# Evaluate CodeBERT on `setup_py_dataset`

Input on Google Drive:
- `NT230/data/d1/saved_models/checkpoint-best-acc/model.bin`
- `NT230/data/setup_py_dataset/benign/*/setup.py`
- `NT230/data/setup_py_dataset/malicious/*/setup.py`

This notebook does not use LLM agents. It evaluates the fine-tuned CodeBERT model directly on each `setup.py` sample.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, zipfile, shutil
from pathlib import Path
from collections import Counter

DRIVE_ROOT = Path('/content/drive/My Drive/NT230')
MODEL_PATH = DRIVE_ROOT / 'data/d1/saved_models/checkpoint-best-acc/model.bin'
DATASET_DIR = DRIVE_ROOT / 'data/setup_py_dataset'
ZIP_PATH = DRIVE_ROOT / 'data/setup_py_dataset.zip'
OUTPUT_DIR = DATASET_DIR / 'results_codebert'

# If only the zip exists on Drive, extract it before evaluation.
# Expected zip content: setup_py_dataset/benign/... and setup_py_dataset/malicious/...
if not DATASET_DIR.exists() and ZIP_PATH.exists():
    print('Extracting:', ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DRIVE_ROOT / 'data')

# Handle zip created with an extra nested setup_py_dataset directory.
nested = DATASET_DIR / 'setup_py_dataset'
if nested.exists() and not (DATASET_DIR / 'benign').exists():
    print('Using nested dataset folder:', nested)
    DATASET_DIR = nested
    OUTPUT_DIR = DATASET_DIR / 'results_codebert'

assert MODEL_PATH.exists(), f'Missing model: {MODEL_PATH}'
assert DATASET_DIR.exists(), f'Missing dataset folder and zip not found: {DATASET_DIR} / {ZIP_PATH}'
assert (DATASET_DIR / 'benign').exists(), f'Missing benign folder: {DATASET_DIR / "benign"}'
assert (DATASET_DIR / 'malicious').exists(), f'Missing malicious folder: {DATASET_DIR / "malicious"}'

print('Model:', MODEL_PATH)
print('Dataset:', DATASET_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
!pip install -q transformers==4.40.0 torch scikit-learn scipy pandas tqdm

In [ ]:
REPO_DIR = Path('/content/NT230')
if not REPO_DIR.exists():
    !git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo ready:', REPO_DIR)

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')

In [ ]:
from lamps.agents.classifier import ClassifierAgent
from lamps.agents.extractor import ExtractedFile
from lamps.evaluation.metrics import classification_report, format_report

classifier = ClassifierAgent(
    checkpoint=str(MODEL_PATH),
    batch_size=64,
)
print('Classifier loaded')

In [ ]:
LABELS = {'benign': 0, 'malicious': 1}
records = []

for label_name, target in LABELS.items():
    label_dir = DATASET_DIR / label_name
    for setup_path in sorted(label_dir.rglob('setup.py')):
        sample_id = setup_path.parent.name
        source = setup_path.read_text(encoding='utf-8', errors='ignore')
        if not source.strip():
            continue
        records.append({
            'sample_id': sample_id,
            'label_name': label_name,
            'target': target,
            'path': setup_path,
            'source': source,
        })

counts = Counter(r['label_name'] for r in records)
print('Samples:', len(records))
print('Counts:', dict(counts))
assert records, 'No setup.py records found'

In [ ]:
from tqdm import tqdm

files = [
    ExtractedFile(
        package=r['sample_id'],
        path=r['path'],
        rel_path='setup.py',
        source=r['source'],
    )
    for r in records
]

print(f'Classifying {len(files)} setup.py files...')
classifications = classifier.classify_files(files)

y_true = [int(r['target']) for r in records]
y_pred = [int(c.target) for c in classifications]
report = classification_report(y_true, y_pred)

print('\n=== setup_py_dataset / CodeBERT ===')
print(format_report(report))

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

predictions = []
for r, c in zip(records, classifications):
    predictions.append({
        'sample_id': r['sample_id'],
        'path': str(r['path'].relative_to(DATASET_DIR)),
        'target': int(r['target']),
        'target_label': r['label_name'],
        'predicted': int(c.target),
        'predicted_label': c.label,
        'score': float(c.score),
        'source_chars': len(r['source']),
    })

(OUTPUT_DIR / 'predictions.jsonl').write_text(
    '\n'.join(json.dumps(p, ensure_ascii=False) for p in predictions) + '\n',
    encoding='utf-8',
)
(OUTPUT_DIR / 'report.json').write_text(json.dumps(report.to_dict(), indent=2), encoding='utf-8')
(OUTPUT_DIR / 'report.txt').write_text(format_report(report), encoding='utf-8')
(OUTPUT_DIR / 'summary.json').write_text(json.dumps({
    'dataset': str(DATASET_DIR),
    'model': str(MODEL_PATH),
    'counts': dict(counts),
    'n_samples': len(records),
    'output': str(OUTPUT_DIR),
}, indent=2), encoding='utf-8')

print('Saved to:', OUTPUT_DIR)
print('- predictions.jsonl')
print('- report.json')
print('- report.txt')
print('- summary.json')